# Stage C 03j — freeze panels and run resumable c16 baselines

This notebook runs a bounded, accession-balanced T4 pilot first. The pilot is an operational and exploratory check, not a model-selection result. Set `RUN_FULL_A100=True` only after reconnecting to an A100 to run the uncapped held-out baseline. Both evaluations save atomic progress checkpoints to Drive and resume automatically when this notebook cell is rerun.


In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='ae72fae21ff9a0b50e4fe1d9d642c38c643b4923'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
SOURCE_ROOT='/content/drive/MyDrive/bacteria_titan_v1_ecoli_related_15gbp'
DATASET_NAME='nonoverlap_6mer_v1'
TAXONOMY_MANIFEST=f'{DRIVE_ROOT}/stage_c_dataset/manifests/accession_manifest.parquet'
ACCESSION_MANIFEST=TAXONOMY_MANIFEST
ANI_MEMBERSHIP=f'{DRIVE_ROOT}/stage_c_dataset/manifests/ani99_membership.parquet'
ANI_PAIRS=f'{DRIVE_ROOT}/inputs/ecoli_skani_triangle.tsv'
NCBI_ZIP_DIR=f'{SOURCE_ROOT}/raw/ncbi_dataset_zips'
PILOT_SEGMENTS_PER_ACCESSION=256
RUN_FULL_A100=False  # Change to True only in an A100 runtime
PILOT_CHECKPOINT_EVERY=64
FULL_CHECKPOINT_EVERY=512
EVALUATION_PROGRESS_EVERY=16


In [ ]:
from pathlib import Path
from google.colab import drive
import json, shutil, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/DATASET_NAME
panels=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3/panels'
PROTOCOL=repo/'studies/stage_c_ecoli_medium_deep_memory_v3/protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_medium_deep_memory_v3'
SOURCE_RESUME_AMENDMENT=repo/'studies/stage_c_ecoli_medium_deep_memory_v3/amendments/c16_resumable_baseline_v1.json'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
AMENDMENT_ID='heldout_high_quality_fallback_v1'
amendment=STUDY_ROOT/'amendments'/f'{AMENDMENT_ID}.json'
if not amendment.is_file():
    changes={'panel_policy':{'training':'complete high-quality E. coli only','heldout':'native val/test; complete preferred with high-quality draft fallback','ani99_isolation':'selected heldout groups excluded from all train panels'}}
    subprocess.run(['seqtrainer-titans-stage-c-study','amend','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--amendment-id',AMENDMENT_ID,'--rationale','The frozen validation split has only two complete E. coli ANI99 representatives, below the predeclared eight-accession gate.','--classification','engineering','--expected-impact','Training eligibility is unchanged. Held-out evaluation may include high-quality draft assemblies, so whole-genome claims remain restricted and assembly-level counts must be reported.','--changes',json.dumps(changes,sort_keys=True)],check=True)
RESUME_AMENDMENT_ID='c16_resumable_baseline_v1'
RESUME_AMENDMENT=STUDY_ROOT/'amendments'/f'{RESUME_AMENDMENT_ID}.json'
if not RESUME_AMENDMENT.is_file():
    source=json.loads(SOURCE_RESUME_AMENDMENT.read_text())
    subprocess.run(['seqtrainer-titans-stage-c-study','amend','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--amendment-id',RESUME_AMENDMENT_ID,'--rationale',source['rationale'],'--classification',source['classification'],'--expected-impact',source['expected_impact'],'--changes',json.dumps(source['changes'],sort_keys=True)],check=True)
def run_logged(root,label,command):
    root.mkdir(parents=True,exist_ok=True)
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(root),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        for path in (root/'FAILED.txt',root/'logs'/f'{label}.log'):
            if path.exists(): print(path.read_text(errors='replace')[-20000:])
        raise
def record_once(run_id,artifact,tier,amendments=()):
    marker=STUDY_ROOT/'record_markers'/f'{run_id}.json'
    if marker.exists():
        print('Ledger record already exists:',marker); return
    command=['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id',run_id,'--evidence-tier',tier,'--artifact',str(artifact)]
    for amendment_path in amendments: command.extend(['--protocol-amendment',str(amendment_path)])
    subprocess.run(command,check=True)
    marker.parent.mkdir(parents=True,exist_ok=True)
    marker.write_text(json.dumps({'run_id':run_id,'artifact':str(artifact)},indent=2)+'\n')


In [ ]:
for path in map(Path,(ACCESSION_MANIFEST,ANI_MEMBERSHIP,ANI_PAIRS)):
    if not path.is_file(): raise FileNotFoundError(path)
if not Path(NCBI_ZIP_DIR).is_dir():
    raise FileNotFoundError('Add the bacteria_titan_v1_ecoli_related_15gbp shared-folder shortcut to My Drive: '+NCBI_ZIP_DIR)
if not (panels/'panel_summary.json').is_file():
    run_logged(panels,'freeze_ecoli_panels',['seqtrainer-titans-stage-c-panel','freeze','--dataset-dir',str(dataset),'--accession-manifest',ACCESSION_MANIFEST,'--ani-membership',ANI_MEMBERSHIP,'--ani-pairs',ANI_PAIRS,'--ncbi-zip-dir',NCBI_ZIP_DIR,'--output-dir',str(panels)])
for name in ('e25','e100','e250','e100_additions','validation','test'):
    subprocess.run(['seqtrainer-titans-stage-c-panel','validate','--dataset-dir',str(dataset),'--panel-manifest',str(panels/f'{name}.json')],check=True)
record_once('ecoli_panel_freeze_v1',panels,'engineering')
summary=json.loads((panels/'panel_summary.json').read_text())
print(json.dumps(summary,indent=2,sort_keys=True))
for role in ('validation','test'):
    panel=json.loads((panels/f'{role}.json').read_text())
    print(role,'accessions=',len(panel['accessions']),'ANI99=',len(set(panel['ani99_groups'])),'assembly_levels=',panel['eligibility']['selected_assembly_level_counts'])


In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime for c16 evaluation.')
gpu_name=torch.cuda.get_device_name(0)
baseline=Path(DRIVE_ROOT)/'runs/c17_v3_c16_broad_baseline_resumable'
pilot=baseline/'pilot_evaluation'
full=baseline/'full_evaluation'
c16=Path(DRIVE_ROOT)/'runs/c16_deep_adaptive_5m_paper_exact/latest.pt'
common=['seqtrainer-titans-stage-c-evaluate','--dataset-dir',str(dataset),'--panel-manifest',str(panels/'validation.json'),'--run',f'c16={c16}','--split','val','--comparison-mode','partial','--device','cuda','--resume','--progress-every-segments',str(EVALUATION_PROGRESS_EVERY),'--protocol',str(PROTOCOL)]
print('GPU:',gpu_name)
print('Safe to rerun this cell after a disconnect: evaluation resumes from <output>/resume/c16.evaluation.pt')
if not (pilot/'evaluation.json').is_file():
    run_logged(baseline,'evaluate_c16_pilot',[*common,'--output-dir',str(pilot),'--max-segments-per-accession',str(PILOT_SEGMENTS_PER_ACCESSION),'--checkpoint-every-segments',str(PILOT_CHECKPOINT_EVERY),'--protocol-amendment',str(RESUME_AMENDMENT),'--run-id','c16_broad_ecoli_pilot_v1'])
record_once('c16_broad_ecoli_pilot_v1',pilot,'exploratory',(RESUME_AMENDMENT,))
pilot_result=json.loads((pilot/'evaluation.json').read_text())
print('PILOT (operational/exploratory only; not a selection result)')
print(json.dumps(pilot_result,indent=2,sort_keys=True))
if RUN_FULL_A100:
    if 'A100' not in gpu_name.upper(): raise RuntimeError('RUN_FULL_A100=True requires an A100 runtime; current GPU is '+gpu_name)
    if not (full/'evaluation.json').is_file():
        run_logged(baseline,'evaluate_c16_full',[*common,'--output-dir',str(full),'--checkpoint-every-segments',str(FULL_CHECKPOINT_EVERY),'--run-id','c16_broad_ecoli_baseline_v1'])
    if not shutil.which('prodigal'):
        subprocess.run(['apt-get','update'],check=True); subprocess.run(['apt-get','install','-y','prodigal'],check=True)
    generation=baseline/'generation_t0p6'
    if not (generation/'generation_evaluation.json').is_file():
        run_logged(baseline,'generate_c16',['seqtrainer-titans-stage-c-generate','--dataset-dir',str(dataset),'--panel-manifest',str(panels/'validation.json'),'--taxonomy-manifest',TAXONOMY_MANIFEST,'--checkpoint',str(c16),'--output-dir',str(generation),'--split','val','--species','Escherichia coli','--prompts','4','--prompt-tokens','128','--new-tokens','1024','--temperatures','0.6','--top-k','1024','--top-p','0.99','--device','cuda','--memory-mode','adaptive','--prodigal',shutil.which('prodigal'),'--protocol',str(PROTOCOL),'--run-id','c16_broad_ecoli_baseline_v1'])
    record_once('c16_broad_ecoli_baseline_v1',baseline,'exploratory')
else:
    print('Pilot complete. For the uncapped baseline, reconnect with an A100, set RUN_FULL_A100=True, and rerun all cells. The full path is independently resumable.')
print('LIVE PILOT STATUS:',pilot/'c16_LIVE_STATUS.json')
print('LIVE FULL STATUS:',full/'c16_LIVE_STATUS.json')
print('SHARE THIS DIRECTORY:',baseline)
